In [ ]:
 #Extract Parquet files from the TLC website

# download_fhvhv
from urllib.request import urlretrieve

years_list =  ['2024']
months_list = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

for year in years_list:
    for month in months_list:
        filename = f"fhvhv_tripdata_{year}-{month}.parquet"
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
        try:
            print(f"Downloading {filename}")

            urlretrieve(url, filename)

            print(f"{filename} downloaded successfully.")

        except Exception as e:
            print(f"Failed to download {filename}: {e}")


In [ ]:
## Exploratory Data Analysis Function
#The purpose of EDA is to understand the FHVHS data at a detailed level. The EDA:
#Determines the schema of the dataset, including the name and data type of each column.
#Calculates the minimum, maximum, mean, and standard deviation for all float, double, and integer type columns.
#Plots the top 10 categories and frequencies for categorical variables.
#For columns with date and datetime data types, identifies the format (e.g., MM/DD/YYYY or YYYY-MM-DD), and calculates the minimum and maximum values.
#Counts the number of missing values for each column.
#Identifies outliers.


# Import the storage module and libraries
from google.cloud import storage
from io import StringIO, BytesIO
import pandas as pd
import dask.dataframe as dd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Set display format
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 1000)

#Define Genneral EDA Function
#Determines the schema of the dataset, including the name and data type of each column.
def perform_EDA(df : pd.DataFrame, filename : str):
    """
    perform_EDA(df : pd.DataFrame, filename : str)
    Accepts a dataframe and a text filename as inputs.
    Runs some basic statistics on the data and outputs to console.

    :param df: The Pandas dataframe to explore
    :param filename: The name of the data file
    :returns:
    """
    print(f"{filename} Number of records:")
    print(df.count())
    number_of_duplicate_records = df.duplicated().sum()
    print(f"{filename} Number of duplicate records: {number_of_duplicate_records}" )
    print(f"{filename} Info")
    print(df.info())
    print(f"{filename} Describe")
    print(df.describe())
    print(f"{filename} Columns with null values")
    print(df.columns[df.isnull().any()].tolist())
    rows_with_null_values = df.isnull().any(axis=1).sum()
    print(f"{filename} Number of Rows with null values: {rows_with_null_values}" )
    integer_column_list = df.select_dtypes(include='int64').columns
    print(f"{filename} Integer data type columns: {integer_column_list}")
    float_column_list = df.select_dtypes(include='float64').columns
    print(f"{filename} Float data type columns: {float_column_list}")

#Define Numerical EDA Sub-Function
def perform_EDA_numeric(df : pd.DataFrame, filename : str):
    """
    perform_EDA_numeric(df : pd.DataFrame, filename : str)
    Accepts a dataframe and a text filename as inputs.
    Runs some basic statistics on numeric columns and saves the output in a dataframe.

    :param df: The Pandas dataframe to explore
    :param filename: The name of the data file
    :returns:
    :   pd.DataFrame: A new dataframe with summary statistics
    """
    # Initialize a list to collect summary data
    summary_data = []
    # Gather summary statistics on numeric columns
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
       summary_data.append({
            'Filename': filename,         'Column': col,
            'Minimum': df[col].min(),     'Maximum': df[col].max(),
            'Average': df[col].mean(),    'Standard Deviation': df[col].std(),
            'Missing Values': df[col].isnull().sum()
        })
    # Convert the summary data list into a DataFrame
    return pd.DataFrame(summary_data)

#Define Categorial Sub-Function
def perform_EDA_categorical(df: pd.DataFrame, filename: str, categorical_columns):
    """
    perform_EDA_categorical(df : pd.DataFrame, filename : str, categorical_columns)
    Accepts a dataframe and a text filename as inputs.
    Collects statistics on categorical columns.

    :param df: The Pandas dataframe to explore
    :param filename: The name of the data file
    :param categorical_columns: A list of column names for categorical columns
    :returns:
    :   pd.DataFrame: A new dataframe with summary statistics
    """
    # Initialize a list to collect summary data
    summary_data = []

    # Getsummary statistics on categorical columns
    for col in categorical_columns:
        summary_data.append({
            'Filename': filename,
            'Column': col,
            'Unique Values': df[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique(),
            'Minimum': df[col].min(),
            'Maximum': df[col].max(),
            'Missing Values': df[col].isnull().sum()
        })

        # Plot top categories and their frequencies
        plt.figure(figsize=(10, 5))
        top_values = df[col].value_counts(normalize=True).head(10)  # Normalize to get proportions
        sns.barplot(x=top_values.index, y=top_values.values, palette="Blues")
        plt.xticks(rotation=45)
        plt.title(f"Top 10 Most Frequent Categories for {col}")
        plt.ylabel("Proportion")
        plt.xlabel(col)
        plt.show()

        # Print top categories and their frequencies
        print(f"\nTop 10 categories for {col}:\n", df[col].value_counts().head(10), "\n")

    # Convert the summary data list into a DataFrame
    return pd.DataFrame(summary_data)

#Define TimeAnalysis Sub-Function
def perform_EDA_time_analysis(df: pd.DataFrame, filename: str, datetime_columns: list):
    """
    perform_EDA_time_analysis(df : pd.DataFrame, filename : str, datetime_columns)
    Accepts a dataframe and a text filename as inputs.
    Collects statistics on time-related columns and performs EDA for time analysis.

    :param df: The Pandas dataframe to explore
    :param filename: The name of the data file
    :param datetime_columns: A list of column names for datetime columns
    :returns:
    :   pd.DataFrame: A new dataframe with summary statistics
    """

    # Convert datetime columns to pandas datetime format
    for col in datetime_columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    # Extract the date and hour from the pickup datetime column
    df['pickup_date'] = df['pickup_datetime'].dt.date  # For day-level analysis
    df['pickup_hour'] = df['pickup_datetime'].dt.hour  # For hour-level analysis

    # Count the number of trips per day
    trip_volume_by_date = df.groupby('pickup_date').size()

    # Count the number of trips per hour of the day
    trip_volume_by_hour = df.groupby('pickup_hour').size()

    # Find the peak day and peak hour
    peak_day = trip_volume_by_date.idxmax()
    peak_day_volume = trip_volume_by_date.max()

    peak_hour = trip_volume_by_hour.idxmax()
    peak_hour_volume = trip_volume_by_hour.max()

    # Plotting the total trip volume over time (by day)
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=trip_volume_by_date.index, y=trip_volume_by_date.values, marker='o')
    plt.title(f"Total Trip Volume Over Time - {filename}")
    plt.xlabel("Date")
    plt.ylabel("Total Trip Volume")
    plt.xticks(rotation=45)

    # Highlight the peak day on the plot
    plt.axvline(x=peak_day, color='red', linestyle='--', label=f'Peak Day: {peak_day} ({peak_day_volume} trips)')
    plt.legend()
    plt.show()

    # Plotting the trip volume by hour of the day
    plt.figure(figsize=(10, 6))
    sns.barplot(x=trip_volume_by_hour.index, y=trip_volume_by_hour.values, palette="Greens")
    plt.title(f"Trip Volume by Hour of the Day - {filename}")
    plt.xlabel("Hour of the Day")
    plt.ylabel("Total Trip Volume")

    # Highlight the peak hour on the plot
    plt.axvline(x=peak_hour, color='red', linestyle='--', label=f'Peak Hour: {peak_hour} ({peak_hour_volume} trips)')
    plt.legend()
    plt.show()

    # Print peak times for reference
    print(f"Peak Day: {peak_day} with {peak_day_volume} trips")
    print(f"Peak Hour: {peak_hour} with {peak_hour_volume} trips")

    # Returning the summary statistics in a dataframe
    summary_timedata = {
        'Peak Day': peak_day,
        'Peak Day Volume': peak_day_volume,
        'Peak Hour': peak_hour,
        'Peak Hour Volume': peak_hour_volume
    }

    return pd.DataFrame([summary_timedata])

# Define Main function for FHVHV
def main_taxi():

    # Categorical columns to analyze
    categorical_columns_list = [
        "PULocationID", "DOLocationID", "shared_request_flag", "shared_match_flag",
        "access_a_ride_flag", "wav_request_flag", "wav_match_flag"
    ]

    # Datetime columns to analyze
    datetime_columns_list = [
        "request_datetime", "on_scene_datetime", "pickup_datetime",
        "dropoff_datetime", "trip_time"
    ]

    # Set bucket name
    bucket_name = 'my-bigdata-project-kc'

    # Create a client object that points to GCS
    storage_client = storage.Client()

    # Get a list of the 'blobs' (objects or files) in the bucket
    blobs = storage_client.list_blobs(bucket_name, prefix="landing/")

    # Iterate through the list and print out their names
    parquet_blobs = [blob for blob in blobs if blob.name.endswith('.parquet')]

    for blob in parquet_blobs:
        print(f"file {blob.name} with size {blob.size} bytes created on {blob.time_created}")

        # Read in the Parquet file from the blob
        # Note the use of BytesIO and .download_as_bytes() function
        df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

        # Perform EDA on the data
        perform_EDA(df, blob.name)

        # Gather the statistics on numeric columns
        numeric_summary_df = perform_EDA_numeric(df, blob.name)
        print(numeric_summary_df.head(24))

        # Gather statistics on the categorical columns
        categorical_summary_df = perform_EDA_categorical(df, blob.name, categorical_columns_list)
        print(categorical_summary_df.head(24))

        # Gather statistics on the datetime columns
        datetime_summary_df = perform_EDA_time_analysis(df, blob.name, datetime_columns_list)
        print(datetime_summary_df.head(24))

        # For testing one file
        if blob.name == "landing/yellow_tripdata_2022-01.parquet":
            break

# Call the main function
if __name__ == "__main__":
    main_taxi()

